The point of this notebook is to create a few datasets that can trigger some tests we are missing or are under represented in the fine-tuning data.
The dataset represents a synthetic clinical trial in which patients are enrolled into one of three treatment arms (A, B, C) and evaluated using a set of biomarkers, clinical outcomes, and categorical health indicators.

Its structure mirrors a realistic early-phase oncology or chronic-disease study, where investigators routinely compare:
Treatment groups
Sex-based differences
Biomarkers with different statistical properties
Continuous physiological measurements
Binary clinical endpoints (response, rare adverse events)

In [1]:
import numpy as np
import pandas as pd
import pathlib

np.random.seed(42)

n = 400
patient_id = np.arange(1, n + 1)

# 3-arm trial: A, B, C
treatment = np.random.choice(["A", "B", "C"], size=n, p=[0.3, 0.4, 0.3])
sex = np.random.choice(["female", "male"], size=n, p=[0.5, 0.5])

df_clinical = pd.DataFrame({
    "patient_id": patient_id,
    "treatment": treatment,
    "sex": sex,
})

# --- t-test targets ---

# Student t-test: groups A vs B, normal, similar variance
mu_A, mu_B = 0.0, 0.6
sd_equal = 1.0

vals_student = []
for t in df_clinical["treatment"]:
    if t == "A":
        vals_student.append(np.random.normal(mu_A, sd_equal))
    elif t == "B":
        vals_student.append(np.random.normal(mu_B, sd_equal))
    else:  # C
        vals_student.append(np.random.normal(mu_A, sd_equal))

df_clinical["biomarker_student_equal_var"] = vals_student

# Welch t-test: groups A vs B, normal, unequal variance
vals_welch = []
for t in df_clinical["treatment"]:
    if t == "A":
        vals_welch.append(np.random.normal(0.0, 1.0))
    elif t == "B":
        vals_welch.append(np.random.normal(0.6, 3.0))
    else:  # C
        vals_welch.append(np.random.normal(0.0, 1.0))

df_clinical["biomarker_welch_unequal_var"] = vals_welch

# --- ANOVA targets ---

# One-way ANOVA: 3 groups, equal variance
anova_equal = []
for t in df_clinical["treatment"]:
    if t == "A":
        anova_equal.append(np.random.normal(0.0, 1.0))
    elif t == "B":
        anova_equal.append(np.random.normal(0.5, 1.0))
    else:  # C
        anova_equal.append(np.random.normal(1.0, 1.0))

df_clinical["outcome_anova_equal_var"] = anova_equal

# Welch ANOVA: 3 groups, unequal variances
anova_unequal = []
for t in df_clinical["treatment"]:
    if t == "A":
        anova_unequal.append(np.random.normal(0.0, 1.0))
    elif t == "B":
        anova_unequal.append(np.random.normal(0.5, 3.0))
    else:  # C
        anova_unequal.append(np.random.normal(1.0, 5.0))

df_clinical["outcome_anova_unequal_var"] = anova_unequal

# --- Correlation targets ---

# Pearson
x_norm = np.random.normal(0, 1, size=n)
noise = np.random.normal(0, 0.5, size=n)
y_norm_linear = 0.8 * x_norm + noise

df_clinical["x_norm"] = x_norm
df_clinical["y_norm_linear"] = y_norm_linear

# Spearman-like
x_skew = np.random.exponential(scale=1.0, size=n)
y_monotonic = np.log1p(x_skew) + np.random.normal(0, 0.2, size=n)

df_clinical["x_skew"] = x_skew
df_clinical["y_monotonic"] = y_monotonic

# --- Categorical for chi-square / Fisher ---

# Chi-square candidate
probs_resp = []
for t in df_clinical["treatment"]:
    if t == "A":
        probs_resp.append(0.20)
    elif t == "B":
        probs_resp.append(0.60)
    else:  # C
        probs_resp.append(0.40)

df_clinical["responder"] = (np.random.rand(n) < np.array(probs_resp)).astype(int)

# Fisher candidate (rare)
probs_rare = []
for t in df_clinical["treatment"]:
    if t == "A":
        probs_rare.append(0.002)
    elif t == "B":
        probs_rare.append(0.01)
    else:
        probs_rare.append(0.05)

df_clinical["rare_event"] = (np.random.rand(n) < np.array(probs_rare)).astype(int)

# --- Save dataset ---

output_dir = pathlib.Path("toy_datasets")
output_dir.mkdir(exist_ok=True)

output_path = output_dir / "synthetic_clinical_study.csv"
df_clinical.to_csv(output_path, index=False)

print(f"Synthetic clinical dataset saved to: {output_path}")
df_clinical.head()


Synthetic clinical dataset saved to: toy_datasets\synthetic_clinical_study.csv


,patient_id,treatment,sex,biomarker_student_equal_var,biomarker_welch_unequal_var,outcome_anova_equal_var,outcome_anova_unequal_var,x_norm,y_norm_linear,x_skew,y_monotonic,responder,rare_event
0,1,B,female,-0.190474,2.168506,1.038296,0.106229,-0.358340,0.065581,0.479636,0.492353,1,0
1,2,C,male,0.471468,-0.573700,2.072507,1.384259,-0.647542,-0.123505,1.652375,1.334728,0,0
2,3,C,male,1.882024,-0.024355,0.635047,-0.124280,0.744192,0.637267,1.735135,1.147397,1,0
3,4,B,male,1.945420,7.026811,-0.339210,-1.450008,-0.181224,0.560250,0.485501,0.347216,1,0
4,5,A,female,1.593187,1.727543,-1.044809,0.168655,-0.649373,-0.314617,0.614565,0.273792,0,0


lets get another dataset:
This dataset simulates an online learning platform running experiments on:
Different learning programs (program: control, video, video+quiz)
Different device types (device: mobile vs desktop)
Student scores, satisfaction, study time, engagement, and rare complaints

In [2]:
import numpy as np
import pandas as pd
import pathlib

np.random.seed(123)

# -----------------------------
# Basic design
# -----------------------------
n = 450
student_id = np.arange(1, n + 1)

program = np.random.choice(
    ["control", "video", "video_quiz"],
    size=n,
    p=[0.3, 0.4, 0.3]
)

device = np.random.choice(
    ["desktop", "mobile"],
    size=n,
    p=[0.55, 0.45]
)

df_online = pd.DataFrame({
    "student_id": student_id,
    "program": program,
    "device": device,
})

# -----------------------------
# t-test targets (device-based)
# -----------------------------
# Student t-test target:
# exam_score_equal_var: normal, similar variance by device
exam_scores_equal = []
for d in df_online["device"]:
    if d == "desktop":
        # slightly higher mean
        exam_scores_equal.append(np.random.normal(loc=75, scale=8))
    else:  # mobile
        exam_scores_equal.append(np.random.normal(loc=72, scale=8))

df_online["exam_score_equal_var"] = exam_scores_equal

# Welch t-test target:
# exam_score_unequal_var: normal-ish, very different variances by device
exam_scores_unequal = []
for d in df_online["device"]:
    if d == "desktop":
        exam_scores_unequal.append(np.random.normal(loc=75, scale=6))
    else:  # mobile: higher variance
        exam_scores_unequal.append(np.random.normal(loc=73, scale=16))

df_online["exam_score_unequal_var"] = exam_scores_unequal

# -----------------------------
# ANOVA targets (program-based)
# -----------------------------
# One-way ANOVA candidate:
# satisfaction_equal_var: 3 groups, different means, similar variance
satisfaction_equal = []
for p in df_online["program"]:
    if p == "control":
        satisfaction_equal.append(np.random.normal(loc=3.2, scale=0.6))
    elif p == "video":
        satisfaction_equal.append(np.random.normal(loc=3.6, scale=0.6))
    else:  # video_quiz
        satisfaction_equal.append(np.random.normal(loc=4.1, scale=0.6))

df_online["satisfaction_equal_var"] = satisfaction_equal

# Welch ANOVA candidate:
# satisfaction_unequal_var: 3 groups, different means AND different variances
satisfaction_unequal = []
for p in df_online["program"]:
    if p == "control":
        satisfaction_unequal.append(np.random.normal(loc=3.0, scale=0.5))
    elif p == "video":
        satisfaction_unequal.append(np.random.normal(loc=3.8, scale=1.0))
    else:  # video_quiz
        satisfaction_unequal.append(np.random.normal(loc=4.3, scale=1.8))

df_online["satisfaction_unequal_var"] = satisfaction_unequal

# -----------------------------
# Correlation targets
# -----------------------------
# Pearson-friendly: roughly bivariate normal with linear relation
hours_studied = np.random.normal(loc=10, scale=3, size=n)
noise = np.random.normal(loc=0, scale=2, size=n)
final_score = 5 * hours_studied + noise  # strong linear link

df_online["hours_studied_norm"] = hours_studied
df_online["final_score_linear"] = final_score

# Spearman-friendly: skewed + monotonic, not strictly linear
time_on_platform = np.random.exponential(scale=1.5, size=n)  # skewed
engagement_index = np.log1p(time_on_platform) + np.random.normal(0, 0.25, size=n)

df_online["time_on_platform_skew"] = time_on_platform
df_online["engagement_monotonic"] = engagement_index

# -----------------------------
# Binary outcomes for Chi-square / Fisher
# -----------------------------
# Pass/fail outcome with decent counts across program → chi-square candidate
pass_probs = []
for p in df_online["program"]:
    if p == "control":
        pass_probs.append(0.65)
    elif p == "video":
        pass_probs.append(0.75)
    else:  # video_quiz
        pass_probs.append(0.82)

df_online["passed_exam"] = (np.random.rand(n) < np.array(pass_probs)).astype(int)

# Rare complaint → Fisher candidate: very low expected counts
complaint_probs = []
for p in df_online["program"]:
    if p == "control":
        complaint_probs.append(0.005)
    elif p == "video":
        complaint_probs.append(0.01)
    else:  # video_quiz
        complaint_probs.append(0.02)

df_online["rare_complaint"] = (np.random.rand(n) < np.array(complaint_probs)).astype(int)


output_dir = pathlib.Path("toy_datasets")
output_dir.mkdir(exist_ok=True)

output_path = output_dir / "synthetic_online_learning.csv"
df_online.to_csv(output_path, index=False)

print(f"Synthetic online learning dataset saved to: {output_path}")
df_online.head()


Synthetic online learning dataset saved to: toy_datasets\synthetic_online_learning.csv


,student_id,program,device,exam_score_equal_var,exam_score_unequal_var,satisfaction_equal_var,satisfaction_unequal_var,hours_studied_norm,final_score_linear,time_on_platform_skew,engagement_monotonic,passed_exam,rare_complaint
0,1,video,desktop,73.320210,68.686326,3.614590,3.568708,14.298513,70.921169,0.918805,0.561835,1,0
1,2,control,mobile,74.406364,78.177514,2.957624,2.527082,7.972563,39.809405,0.502640,0.147562,1,0
2,3,control,desktop,68.436996,85.821515,2.897849,2.835002,11.593447,57.641266,1.728306,0.985550,0,0
3,4,video,mobile,67.018692,100.711394,4.219248,2.647057,9.030690,40.926040,0.961295,0.606416,1,0
4,5,video_quiz,mobile,74.410324,49.377611,4.644490,8.076113,9.794511,49.814703,1.927612,0.602800,1,0


In [4]:
#another dataset: Factory Quality & Reliability Dataset
#simulated data from a manufacturing plant with two production lines and two machine types

import numpy as np
import pandas as pd
import pathlib

rng = np.random.default_rng(123)

n = 300
batch_id = np.arange(1, n + 1)

# Two production lines and two machine types
line = rng.choice(["line_A", "line_B"], size=n, p=[0.5, 0.5])
shift = rng.choice(["day", "night"], size=n, p=[0.6, 0.4])
machine_type = rng.choice(["legacy", "new"], size=n, p=[0.7, 0.3])

df_factory = pd.DataFrame({
    "batch_id": batch_id,
    "line": line,
    "shift": shift,
    "machine_type": machine_type,
})

# --- t-test targets ---

# Helper masks for line
mask_A = df_factory["line"] == "line_A"
mask_B = ~mask_A

# 1) Student t-test: normal, similar variances across groups
# line_A: mean 100, sd 5
# line_B: mean 103, sd 5
throughput_equal = np.empty(n)
throughput_equal[mask_A] = rng.normal(100, 5, mask_A.sum())
throughput_equal[mask_B] = rng.normal(103, 5, mask_B.sum())
df_factory["throughput_equal"] = throughput_equal

# 2) Welch t-test: normal, unequal variances
# line_A: mean 100, sd 4
# line_B: mean 103, sd 10
throughput_unequal = np.empty(n)
throughput_unequal[mask_A] = rng.normal(100, 4, mask_A.sum())
throughput_unequal[mask_B] = rng.normal(103, 10, mask_B.sum())
df_factory["throughput_unequal"] = throughput_unequal

# --- Correlation targets (designed for Pearson) ---

# 1) temp_setting & pressure_reading: roughly bivariate normal, linear relation
temp_setting = rng.normal(70, 3, size=n)  # around 70°C
noise_pressure = rng.normal(0, 1.5, size=n)
pressure_reading = 2.5 * temp_setting + noise_pressure  # strong linear relationship

df_factory["temp_setting"] = temp_setting
df_factory["pressure_reading"] = pressure_reading

# 2) speed_setting & output_quality_score: another linear, normal-ish pair
speed_setting = rng.normal(50, 4, size=n)  # conveyor speed
noise_quality = rng.normal(0, 2.0, size=n)
output_quality_score = 80 + 0.9 * speed_setting + noise_quality

df_factory["speed_setting"] = speed_setting
df_factory["output_quality_score"] = output_quality_score

# --- Binary outcome for Fisher’s Exact ---

# critical_failure: rare event, more common on new machines
# This is set up so expected counts in a 2x2 table are < 5 in at least one cell.
failure_prob = np.where(df_factory["machine_type"] == "legacy", 0.01, 0.07)
critical_failure = (rng.random(n) < failure_prob).astype(int)
df_factory["critical_failure"] = critical_failure

# Optional sanity check (if you want to inspect):
# print(pd.crosstab(df_factory["machine_type"], df_factory["critical_failure"]))

# --- Save to toy_datasets ---

toy_dir = pathlib.Path("toy_datasets")
toy_dir.mkdir(exist_ok=True)

factory_path = toy_dir / "factory_quality.csv"
df_factory.to_csv(factory_path, index=False)

factory_path


WindowsPath('toy_datasets/factory_quality.csv')

In [6]:
#lets create another one: Employee Wellness & Productivity Dataset, this one will have missing values as well.
import numpy as np
import pandas as pd
import pathlib

rng = np.random.default_rng(2025)

n = 240
employee_id = np.arange(1, n + 1)

# Group variables
office = rng.choice(["onsite", "remote"], size=n, p=[0.6, 0.4])
department = rng.choice(["sales", "engineering", "support"], size=n, p=[0.3, 0.45, 0.25])

df_wellness = pd.DataFrame({
    "employee_id": employee_id,
    "office": office,
    "department": department,
})

# Masks for convenience
mask_onsite = df_wellness["office"] == "onsite"
mask_remote = ~mask_onsite

mask_sales = df_wellness["department"] == "sales"
mask_eng = df_wellness["department"] == "engineering"
mask_support = df_wellness["department"] == "support"

# --- t-test targets ---

# 1) Student's t-test: normal, similar variances
# onsite: mean 75, sd 8; remote: mean 80, sd 8
prod_equal = np.empty(n)
prod_equal[mask_onsite] = rng.normal(75, 8, mask_onsite.sum())
prod_equal[mask_remote] = rng.normal(80, 8, mask_remote.sum())
df_wellness["productivity_equal"] = prod_equal

# 2) Welch's t-test: normal, strongly unequal variances
# onsite: mean 75, sd 6; remote: mean 82, sd 16
prod_unequal = np.empty(n)
prod_unequal[mask_onsite] = rng.normal(75, 6, mask_onsite.sum())
prod_unequal[mask_remote] = rng.normal(82, 16, mask_remote.sum())
df_wellness["productivity_unequal"] = prod_unequal

# --- ANOVA targets ---

# 3) One-way ANOVA: 3 groups, similar variances
# sales: mean 70, sd 7; eng: 75, sd 7; support: 78, sd 7
eng_equal = np.empty(n)
eng_equal[mask_sales] = rng.normal(70, 7, mask_sales.sum())
eng_equal[mask_eng] = rng.normal(75, 7, mask_eng.sum())
eng_equal[mask_support] = rng.normal(78, 7, mask_support.sum())
df_wellness["engagement_equal"] = eng_equal

# 4) Welch ANOVA: 3 groups, very different variances
# sales: mean 70, sd 5; eng: 76, sd 10; support: 82, sd 18
eng_unequal = np.empty(n)
eng_unequal[mask_sales] = rng.normal(70, 5, mask_sales.sum())
eng_unequal[mask_eng] = rng.normal(76, 10, mask_eng.sum())
eng_unequal[mask_support] = rng.normal(82, 18, mask_support.sum())
df_wellness["engagement_unequal"] = eng_unequal

# --- Correlation targets (Pearson) ---

# 5) desk_temp & energy_usage: bivariate normal-ish, strong linear
desk_temp = rng.normal(22, 1.0, size=n)          # °C
noise_energy = rng.normal(0, 5.0, size=n)
energy_usage = 50 + 6 * desk_temp + noise_energy # ~linear
df_wellness["desk_temp"] = desk_temp
df_wellness["energy_usage"] = energy_usage

# 6) steps_per_day & resting_hr: negative relation
steps_per_day = rng.normal(8000, 1500, size=n)
noise_hr = rng.normal(0, 3.0, size=n)
resting_hr = 80 - 0.0015 * steps_per_day + noise_hr
df_wellness["steps_per_day"] = steps_per_day
df_wellness["resting_hr"] = resting_hr

# --- Fisher's Exact target: 2x2 with low expected cell ---

# critical_incident: very rare, slightly more common onsite
# ensure some expected counts < 5
base_prob = np.where(df_wellness["office"] == "onsite", 0.03, 0.01)
critical_incident = (rng.random(n) < base_prob).astype(int)
df_wellness["critical_incident"] = critical_incident

# --- Inject Missing Data (to exercise missing-data pipeline) ---

# ~15% missing in engagement_equal and engagement_unequal
n_miss_eng = int(0.15 * n)
missing_idx_eng = rng.choice(n, size=n_miss_eng, replace=False)
df_wellness.loc[missing_idx_eng, "engagement_equal"] = np.nan

missing_idx_eng2 = rng.choice(n, size=n_miss_eng, replace=False)
df_wellness.loc[missing_idx_eng2, "engagement_unequal"] = np.nan

# ~10% missing in productivity_unequal
n_miss_prod = int(0.10 * n)
missing_idx_prod = rng.choice(n, size=n_miss_prod, replace=False)
df_wellness.loc[missing_idx_prod, "productivity_unequal"] = np.nan

# (Keep correlation vars and critical_incident fully observed so those pipelines run cleanly.)

# --- Save to toy_datasets ---

toy_dir = pathlib.Path("toy_datasets")
toy_dir.mkdir(exist_ok=True)

wellness_path = toy_dir / "wellness_productivity.csv"
df_wellness.to_csv(wellness_path, index=False)

wellness_path


WindowsPath('toy_datasets/wellness_productivity.csv')

In [7]:
#anohter dataset: Housing & Neighborhood Well-Being Survey
import numpy as np
import pandas as pd
import pathlib

rng = np.random.default_rng(2027)

n = 320
household_id = np.arange(1, n + 1)

# Categorical base variables
region = rng.choice(["urban", "suburban", "rural"], size=n, p=[0.4, 0.4, 0.2])
heating_type = rng.choice(["gas", "electric", "other"], size=n, p=[0.5, 0.35, 0.15])
ownership = rng.choice(["owner", "renter"], size=n, p=[0.6, 0.4])
has_pet = rng.choice(["yes", "no"], size=n, p=[0.7, 0.3])

df_housing = pd.DataFrame({
    "household_id": household_id,
    "region": region,
    "heating_type": heating_type,
    "ownership": ownership,
    "has_pet": has_pet,
})

# Convenience masks
mask_owner = df_housing["ownership"] == "owner"
mask_renter = ~mask_owner

mask_urban = df_housing["region"] == "urban"
mask_suburban = df_housing["region"] == "suburban"
mask_rural = df_housing["region"] == "rural"

mask_gas = df_housing["heating_type"] == "gas"
mask_electric = df_housing["heating_type"] == "electric"
mask_other = df_housing["heating_type"] == "other"

# --- t-test targets ---

# 1) Student t-test target: roughly equal variances
# Owners slightly lower stress, renters slightly higher; both normal, same sd
stress_equal = np.empty(n)
stress_equal[mask_owner] = rng.normal(45, 8, mask_owner.sum())
stress_equal[mask_renter] = rng.normal(50, 8, mask_renter.sum())
df_housing["stress_equal"] = stress_equal

# 2) Welch t-test target: unequal variances
# Owners: mean 45, sd 6; renters: mean 55, sd 15
stress_unequal = np.empty(n)
stress_unequal[mask_owner] = rng.normal(45, 6, mask_owner.sum())
stress_unequal[mask_renter] = rng.normal(55, 15, mask_renter.sum())
df_housing["stress_unequal"] = stress_unequal

# --- ANOVA targets ---

# 3) One-way ANOVA target: similar variances across regions
# Satisfaction on 0–100 scale
# urban: 70, suburban: 75, rural: 78, sd ~ 7 everywhere
sat_equal = np.empty(n)
sat_equal[mask_urban] = rng.normal(70, 7, mask_urban.sum())
sat_equal[mask_suburban] = rng.normal(75, 7, mask_suburban.sum())
sat_equal[mask_rural] = rng.normal(78, 7, mask_rural.sum())
df_housing["home_satisfaction_equal"] = sat_equal

# 4) Welch ANOVA target: unequal variances by heating type
# gas: mean 90, sd 5; electric: mean 85, sd 10; other: mean 80, sd 18
sat_unequal = np.empty(n)
sat_unequal[mask_gas] = rng.normal(90, 5, mask_gas.sum())
sat_unequal[mask_electric] = rng.normal(85, 10, mask_electric.sum())
sat_unequal[mask_other] = rng.normal(80, 18, mask_other.sum())
df_housing["home_satisfaction_unequal"] = sat_unequal

# --- Correlation targets (aiming for Pearson) ---

# 5) indoor_temp vs energy_kwh: linear relation, approximately normal
indoor_temp = rng.normal(21.5, 1.2, size=n)  # °C
noise_energy = rng.normal(0, 8.0, size=n)
energy_kwh = 150 + 10 * (indoor_temp - 20) + noise_energy
df_housing["indoor_temp"] = indoor_temp
df_housing["energy_kwh"] = energy_kwh

# 6) sleep_hours vs stress_equal: negative linear relation
sleep_hours = rng.normal(7.0, 1.0, size=n)
noise_stress = rng.normal(0, 3.0, size=n)
# Construct a variant of stress linked to sleep for correlation
stress_for_corr = 55 - 4 * (sleep_hours - 7) + noise_stress
df_housing["sleep_hours"] = sleep_hours
df_housing["stress_for_corr"] = stress_for_corr

# --- Fisher's Exact target: rare binary outcome ---

# severe_allergy: very rare, slightly more common in pet households
# design so that some expected counts < 5 in the 2x2 table
prob_allergy = np.where(df_housing["has_pet"] == "yes", 0.03, 0.002)
severe_allergy = (rng.random(n) < prob_allergy).astype(int)
df_housing["severe_allergy"] = severe_allergy

# --- Inject Missing Data (numeric & categorical) ---

def add_missing(series, frac, rng):
    series = series.copy()
    k = int(frac * len(series))
    if k <= 0:
        return series
    idx = rng.choice(len(series), size=k, replace=False)
    series.iloc[idx] = np.nan
    return series

# Numeric missingness:
# - Low (~5%) in stress_equal
# - High (~35%) in stress_unequal
# - Medium (~20%) in home_satisfaction_equal
# - High (~45%) in home_satisfaction_unequal
# - Low (~8%) in indoor_temp
# - Medium (~25%) in sleep_hours
df_housing["stress_equal"] = add_missing(df_housing["stress_equal"], 0.05, rng)
df_housing["stress_unequal"] = add_missing(df_housing["stress_unequal"], 0.35, rng)
df_housing["home_satisfaction_equal"] = add_missing(df_housing["home_satisfaction_equal"], 0.20, rng)
df_housing["home_satisfaction_unequal"] = add_missing(df_housing["home_satisfaction_unequal"], 0.45, rng)
df_housing["indoor_temp"] = add_missing(df_housing["indoor_temp"], 0.08, rng)
df_housing["sleep_hours"] = add_missing(df_housing["sleep_hours"], 0.25, rng)
# Leave energy_kwh & stress_for_corr mostly complete to keep correlations usable

# Categorical missingness:
# - Some missing in region (medium ~20%)
# - Some missing in heating_type (medium-high ~30%)
# - Some missing in ownership (low ~10%)
# - Some missing in has_pet (medium ~25%)
df_housing["region"] = add_missing(df_housing["region"], 0.20, rng)
df_housing["heating_type"] = add_missing(df_housing["heating_type"], 0.30, rng)
df_housing["ownership"] = add_missing(df_housing["ownership"], 0.10, rng)
df_housing["has_pet"] = add_missing(df_housing["has_pet"], 0.25, rng)

# Leave severe_allergy fully observed so the 2x2 is defined;
# rows with missing has_pet will be dropped by your pipeline as needed.

# --- Save to toy_datasets ---

toy_dir = pathlib.Path("toy_datasets")
toy_dir.mkdir(exist_ok=True)

housing_path = toy_dir / "housing_missingness.csv"
df_housing.to_csv(housing_path, index=False)

housing_path


WindowsPath('toy_datasets/housing_missingness.csv')

In [8]:
#another dataset: E-commerce Customer Behavior & Loyalty

import numpy as np
import pandas as pd
import pathlib

np.random.seed(2026)

n = 500
customer_id = np.arange(1, n + 1)

# Core categorical features
region = np.random.choice(["NA", "EU", "APAC"], size=n, p=[0.4, 0.35, 0.25])
device = np.random.choice(["desktop", "mobile", "tablet"], size=n, p=[0.45, 0.45, 0.10])
membership_level = np.random.choice(["free", "standard", "premium"], size=n, p=[0.45, 0.35, 0.20])
gender = np.random.choice(["female", "male"], size=n, p=[0.5, 0.5])
campaign_source = np.random.choice(["organic", "email", "ads"], size=n, p=[0.5, 0.25, 0.25])

# Binary premium flag derived from membership
premium_flag = (membership_level == "premium").astype(int)

# -----------------------------
# Numeric variables for t-tests
# -----------------------------

# monthly_spend_equal:
# - All groups roughly normal
# - premium has higher mean, but similar variance → Student t candidate
spend_equal = []
for lvl in membership_level:
    if lvl == "free":
        spend_equal.append(np.random.normal(40, 10))
    elif lvl == "standard":
        spend_equal.append(np.random.normal(70, 10))
    else:  # premium
        spend_equal.append(np.random.normal(110, 10))
spend_equal = np.array(spend_equal)

# monthly_spend_unequal:
# - same mean pattern but very different variances → Welch t / Welch ANOVA candidate
spend_unequal = []
for lvl in membership_level:
    if lvl == "free":
        spend_unequal.append(np.random.normal(40, 8))
    elif lvl == "standard":
        spend_unequal.append(np.random.normal(70, 15))
    else:  # premium
        spend_unequal.append(np.random.normal(110, 30))
spend_unequal = np.array(spend_unequal)

# session_duration_equal (minutes per visit):
# depends on device, but similar variance
session_equal = []
for d in device:
    if d == "desktop":
        session_equal.append(np.random.normal(12, 2.5))
    elif d == "mobile":
        session_equal.append(np.random.normal(9, 2.5))
    else:  # tablet
        session_equal.append(np.random.normal(11, 2.5))
session_equal = np.array(session_equal)

# session_duration_unequal:
# same mean pattern, very different variance → Welch ANOVA candidate
session_unequal = []
for d in device:
    if d == "desktop":
        session_unequal.append(np.random.normal(12, 2))
    elif d == "mobile":
        session_unequal.append(np.random.normal(9, 5))
    else:  # tablet
        session_unequal.append(np.random.normal(11, 7))
session_unequal = np.array(session_unequal)

# -----------------------------
# Satisfaction & loyalty (ANOVA targets + correlations)
# -----------------------------

# satisfaction_equal: 1–100 scale
# depends on membership; similar variance
satisfaction_equal = []
for lvl in membership_level:
    if lvl == "free":
        satisfaction_equal.append(np.random.normal(65, 8))
    elif lvl == "standard":
        satisfaction_equal.append(np.random.normal(75, 8))
    else:
        satisfaction_equal.append(np.random.normal(85, 8))
satisfaction_equal = np.array(satisfaction_equal)

# satisfaction_unequal: also affected by campaign_source with unequal variances
satisfaction_unequal = []
for lvl, camp in zip(membership_level, campaign_source):
    base_mean = 65
    if lvl == "standard":
        base_mean = 75
    elif lvl == "premium":
        base_mean = 85

    # variance depends heavily on campaign
    if camp == "organic":
        sd = 6
    elif camp == "email":
        sd = 10
    else:  # ads
        sd = 14

    satisfaction_unequal.append(np.random.normal(base_mean, sd))
satisfaction_unequal = np.array(satisfaction_unequal)

# loyalty_index: linear combination of spend_equal and satisfaction_equal
noise_loyalty = np.random.normal(0, 5, size=n)
loyalty_index = 0.3 * spend_equal + 0.5 * satisfaction_equal + noise_loyalty

# -----------------------------
# Binary outcomes for Fisher / chi-square
# -----------------------------

# High refund frequency flag (~10–15%), more common in "ads" and "mobile"
refund_prob = []
for camp, d in zip(campaign_source, device):
    base = 0.05
    if camp == "ads":
        base += 0.05
    if d == "mobile":
        base += 0.03
    refund_prob.append(base)
refund_prob = np.array(refund_prob)
high_refund_flag = (np.random.rand(n) < refund_prob).astype(int)

# Chargebacks: rare event → Fisher candidate vs premium_flag
# non-premium: ~1%, premium: ~4%
cb_prob = np.where(premium_flag == 1, 0.04, 0.01)
chargeback_flag = (np.random.rand(n) < cb_prob).astype(int)

# Another binary: high_loyalty_flag (top 30% of loyalty_index)
high_loyalty_flag = (loyalty_index >= np.percentile(loyalty_index, 70)).astype(int)

# -----------------------------
# Assemble DataFrame
# -----------------------------

df_ecom = pd.DataFrame({
    "customer_id": customer_id,
    "region": region,
    "device": device,
    "membership_level": membership_level,
    "gender": gender,
    "campaign_source": campaign_source,
    "premium_flag": premium_flag,
    "monthly_spend_equal": spend_equal,
    "monthly_spend_unequal": spend_unequal,
    "session_duration_equal": session_equal,
    "session_duration_unequal": session_unequal,
    "satisfaction_equal": satisfaction_equal,
    "satisfaction_unequal": satisfaction_unequal,
    "loyalty_index": loyalty_index,
    "high_refund_flag": high_refund_flag,
    "chargeback_flag": chargeback_flag,
    "high_loyalty_flag": high_loyalty_flag,
})

# -----------------------------
# Inject missingness
# -----------------------------

rng = np.random.default_rng(777)

# Low missingness (~5%) in region and session_duration_equal
mask_region_low = rng.random(n) < 0.05
mask_session_low = rng.random(n) < 0.05
df_ecom.loc[mask_region_low, "region"] = np.nan
df_ecom.loc[mask_session_low, "session_duration_equal"] = np.nan

# Medium missingness (~20%) in membership_level and monthly_spend_unequal
mask_membership_med = rng.random(n) < 0.20
mask_spend_med = rng.random(n) < 0.20
df_ecom.loc[mask_membership_med, "membership_level"] = np.nan
df_ecom.loc[mask_spend_med, "monthly_spend_unequal"] = np.nan

# High missingness (~30–35%) in satisfaction_unequal and loyalty_index
mask_sat_high = rng.random(n) < 0.35
mask_loyalty_high = rng.random(n) < 0.30
df_ecom.loc[mask_sat_high, "satisfaction_unequal"] = np.nan
df_ecom.loc[mask_loyalty_high, "loyalty_index"] = np.nan

# Some missingness in chargeback_flag to exercise missing handling on binary outcome
mask_cb = rng.random(n) < 0.15
df_ecom.loc[mask_cb, "chargeback_flag"] = np.nan

# Save to toy_datasets
toy_dir = pathlib.Path("toy_datasets")
toy_dir.mkdir(exist_ok=True)
ecom_path = toy_dir / "synthetic_ecommerce_behavior.csv"
df_ecom.to_csv(ecom_path, index=False)

ecom_path


WindowsPath('toy_datasets/synthetic_ecommerce_behavior.csv')